## Section 0 

In [27]:
import subprocess, sys
def _pip(*a):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *a], check=False)
import importlib
for m in ["pyarrow"]:
    try: importlib.import_module(m)
    except Exception: _pip(m)
print("base setup done")

base setup done


In [28]:
import os, glob, json, warnings, itertools
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.preprocessing import MinMaxScaler
from scipy import stats

SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
WORK = "/kaggle/working" if os.path.isdir("/kaggle") else os.path.abspath("../outputs")
os.makedirs(WORK, exist_ok=True)

HOLDOUT = 12
LOOKBACK = 12
MIN_ZONE_MONTHS = 36         
ALPHA_MAX = 0.5              

MODELS = ["SARIMA", "Nexus"]   
RESOURCES = {"electricity": "zone", "carbon": "zone", "water": "basin"}  

def section(t): print("\n" + "=" * 78 + f"\n{t}\n" + "=" * 78)
print("torch", torch.__version__, "| device", DEVICE, "| SEED", SEED)
print("MODELS (run order):", MODELS)
print("RESOURCES (region type):", RESOURCES)

torch 2.10.0+cpu | device cpu | SEED 42
MODELS (run order): ['SARIMA', 'Nexus']
RESOURCES (region type): {'electricity': 'zone', 'carbon': 'zone', 'water': 'basin'}


In [29]:

BASIN_MAX_KM = 2500
BASIN_CENTROIDS = {
    # Africa
    "Congo":(-3,23),"Nile":(20,31),"Niger":(12,4),"Zambezi":(-14,30),"Lake Chad":(13,15),
    "Limpopo":(-23,29),"Orange":(-29,22),"Okavango":(-19,22),"Cuanza":(-10,16),"Ogooue":(-0.7,11),
    "Sanaga":(5,12),"Rovuma":(-11,38),"Rufiji":(-8,37),"Shebelle":(4,44),"Lake Turkana":(4,36),
    "Senegal":(15,-14),"Volta":(9,-1),
    # Europe
    "Danube":(47,20),"Rhine":(50,7),"Loire":(47,1),"Elbe River":(52,11),"Oder River":(52,15),
    "Wisla":(52,20),"Volga":(53,45),"Dniepr":(50,32),"Don":(49,42),"Neva":(60,31),
    "Northern Dvina(Severnaya Dvina)":(62,44),"Pechora":(66,54),"Ural":(50,53),
    # North America
    "Mississippi River":(37,-92),"Colorado River (Pacific Ocean)":(36,-111),"Columbia River":(46,-119),
    "Brazos River":(31,-97),"Bravo":(29,-102),"St.Lawrence":(46,-75),"Mackenzie River":(64,-124),
    "Nelson River":(54,-98),"Churchill River":(56,-95),"Fraser River":(53,-122),"Yukon River":(64,-150),
    "Kuskokwim River":(62,-158),"Albany River":(51,-84),"Nottaway":(50,-77),"Back River":(66,-96),
    "Thelon River":(64,-101),"Santiago":(21,-104),"Grisalva":(17,-93),
    # South America
    "Amazonas":(-4,-62),"Parana":(-24,-56),"Orinoco":(6,-66),"Sao Francisco":(-10,-42),"Tocantins":(-8,-49),
    "Magdalena":(6,-74),"Negro (Argentinia)":(-40,-66),"Colorado (Argentinia)":(-37,-68),"Chubut":(-43,-68),
    "Salado":(-30,-61),"Rio Parnaiba":(-6,-43),"Uruguay":(-31,-57),"Lake Mar Chiquita":(-30,-62),
    # Asia (South / SE / East)
    "Ganges":(26,83),"Brahmaputra":(27,91),"Indus":(28,70),"Godavari":(19,79),"Krishna":(16,78),
    "Mahanadi River (Mahahadi)":(21,84),"Mekong":(17,104),"Salween":(22,98),"Irrawaddy":(22,96),
    "Chao Phraya":(15,100),"Hong(Red River)":(22,104),"Yangtze River (Chang Jiang)":(30,112),
    "Huang He (Yellow River)":(37,110),"Huai He":(33,116),"Xi Jiang":(24,110),"Liao He":(42,122),
    "Yongding He":(40,116),"Amur":(51,127),"Tarim":(40,84),"Balkhash":(46,74),"Issyk-kul":(42,77),
    "Aral Drainage":(44,62),"Tigris & Euphrates":(33,44),"Kura":(41,46),
    # Siberia / Arctic Russia
    "Ob":(60,72),"Yenisei":(62,90),"Lena":(64,126),"Kolyma":(65,155),"Indigirka":(68,146),"Yana":(69,135),
    "Olenek":(70,120),"Khatanga":(72,102),"Taz":(66,80),"Anadyr":(65,173),"Lake Taymur":(74,101),
    # Australia
    "Murray":(-34,144),"Eyre Lake":(-28,137),"Burdekin":(-20,146),"Fitzroy":(-23,150),
}
def _haversine_km(la1, lo1, la2, lo2):
    R = 6371.0088; p = np.pi / 180.0
    dla = (la2 - la1) * p; dlo = (lo2 - lo1) * p
    x = np.sin(dla/2)**2 + np.cos(la1*p)*np.cos(la2*p)*np.sin(dlo/2)**2
    return 2 * R * np.arcsin(np.sqrt(np.clip(x, 0, 1)))
print(f"BASIN_CENTROIDS: {len(BASIN_CENTROIDS)} basin centres | nearest-centroid assignment, cutoff {BASIN_MAX_KM} km.")

BASIN_CENTROIDS: 100 basin centres | nearest-centroid assignment, cutoff 2500 km.


## Section 1

In [30]:
INPUT_ROOT = "/kaggle/input" if os.path.isdir("/kaggle/input") else os.path.abspath("../data")
def find_file(basename_pred, root=INPUT_ROOT):
    for base in [root, WORK]:
        if not os.path.isdir(base): continue
        for r, dirs, files in os.walk(base):
            for f in files:
                if basename_pred(f): return os.path.join(r, f)
            for d in dirs:
                if basename_pred(d): return os.path.join(r, d)
    return None

paths = {
    "aligned":  find_file(lambda f: f == "aligned_dataset.parquet") or
                find_file(lambda f: f.startswith("aligned_dataset") and f.endswith((".parquet", ".csv"))),
    "w_coloc":  find_file(lambda f: f == "gat_weights_colocation.pt"),
    "w_random": find_file(lambda f: f == "gat_weights_random_control.pt"),
    "nodes":    find_file(lambda f: f == "graph_nodes.parquet"),
    "ember_us": find_file(lambda f: f.endswith(".csv") and "us_monthly" in f.lower()),
    "ember_eu": find_file(lambda f: f.endswith(".csv") and "europe_monthly" in f.lower()),
    "ember_in": find_file(lambda f: f.endswith(".csv") and "india_monthly" in f.lower()),
    "g3p_tws":  find_file(lambda f: f.endswith(".csv") and "tws_rivbas" in f.lower()),   # NEW
}
print("Resolved paths:")
for k, v in paths.items():
    print(f"  [{'OK ' if v and os.path.exists(v) else '!! '}] {k:9s} -> {v}")

REQUIRED = ["aligned", "w_coloc", "w_random", "ember_us", "ember_eu", "ember_in", "g3p_tws"]
missing = [k for k in REQUIRED if not (paths.get(k) and os.path.exists(paths[k]))]
if missing:
    raise FileNotFoundError(
        "STOP — required input(s) missing: " + ", ".join(missing) +
        "\n  * aligned/w_coloc/w_random come from gat-colocation-weights-water-energy.ipynb."
        "\n  * ember_* are the EMBER monthly CSVs (US/EU/India)."
        "\n  * g3p_tws is G3P_v1.12_tws_rivbas.csv (see arima-xlstm-water-modelling.ipynb, Part 1)."
        "\nThe notebook will NOT fabricate or substitute a source.")
print("\nAll required inputs located, including the new water series (g3p_tws).")

Resolved paths:
  [OK ] aligned   -> /kaggle/input/notebooks/energyclimaterp/gat-colocation-gnn/aligned_dataset.parquet
  [OK ] w_coloc   -> /kaggle/input/notebooks/energyclimaterp/gat-colocation-gnn/gat_weights_colocation.pt
  [OK ] w_random  -> /kaggle/input/notebooks/energyclimaterp/gat-colocation-gnn/gat_weights_random_control.pt
  [OK ] nodes     -> /kaggle/input/notebooks/energyclimaterp/gat-colocation-gnn/graph_nodes.parquet
  [OK ] ember_us  -> /kaggle/input/datasets/energyclimaterp/us-monthly-ember-electricity/us_monthly_full_release_long_format.csv
  [OK ] ember_eu  -> /kaggle/input/datasets/energyclimaterp/eu-ember-monthly-electricity/europe_monthly_full_release_long_format.csv
  [OK ] ember_in  -> /kaggle/input/datasets/energyclimaterp/india-ember-energy-monthly-electricity/india_monthly_full_release_long_format (1).csv
  [OK ] g3p_tws   -> /kaggle/input/datasets/energyclimaterp/groundwater-dataset/G3P_v1.12_tws_rivbas.csv

All required inputs located, including the new wat

In [31]:

section("INSPECT: aligned dataset (Step 1 checkpoint)")
aligned = (pd.read_parquet(paths["aligned"]) if paths["aligned"].endswith(".parquet")
           else pd.read_csv(paths["aligned"]))
print("shape:", aligned.shape, "\ncolumns:", list(aligned.columns))
print("dtypes:\n", aligned.dtypes.to_string())
for need in ["site_id", "grid_zone_id", "pfaf_id"]:
    if need not in aligned.columns:
        raise KeyError(f"STOP — aligned dataset lacks required column '{need}'. Present: {list(aligned.columns)}")
if "is_node" in aligned.columns:
    print("\nis_node=True rows (graph nodes):", int(aligned["is_node"].sum()))


INSPECT: aligned dataset (Step 1 checkpoint)
shape: (6131, 16) 
columns: ['site_id', 'latitude', 'longitude', 'country_iso3', 'country', 'name_1', 'pfaf_id', 'water_stress_bws', 'grid_zone_id', 'electricity_gwh', 'co2_intensity', 'name', 'company', 'city', 'temperature_c', 'is_node']
dtypes:
 site_id               int64
latitude            float64
longitude           float64
country_iso3         object
country              object
name_1               object
pfaf_id             float64
water_stress_bws    float64
grid_zone_id         object
electricity_gwh     float64
co2_intensity       float64
name                 object
company              object
city                 object
temperature_c       float64
is_node                bool

is_node=True rows (graph nodes): 4887


In [32]:
section("INSPECT: GAT weight files (colocation + random control)")
# import torch_geometric  # unused: the .pt payloads are plain dicts, torch.load is enough

def load_weight_payload(path, label):
    obj = torch.load(path, map_location="cpu", weights_only=False)
    print(f"\n--- {label} :: {path}")
    if not isinstance(obj, dict):
        raise TypeError(f"STOP — {label} is a {type(obj)}, expected a dict payload. Cannot proceed.")
    print("keys:", list(obj.keys()))
    for k, v in obj.items():
        if torch.is_tensor(v):
            print(f"   {k:26s} tensor shape={tuple(v.shape)} dtype={v.dtype}")
        elif isinstance(v, (list, tuple)):
            print(f"   {k:26s} {type(v).__name__} len={len(v)} sample={list(v)[:4]}")
        else:
            print(f"   {k:26s} {type(v).__name__} = {v}")
    return obj

W_COLOC  = load_weight_payload(paths["w_coloc"],  "gat_weights_colocation")
W_RANDOM = load_weight_payload(paths["w_random"], "gat_weights_random_control")

REQ_KEYS = ["site_id", "embeddings", "site_attention_received",
            "attention_edge_index", "attention_edge_weight"]
for label, W in [("colocation", W_COLOC), ("random_control", W_RANDOM)]:
    absent = [k for k in REQ_KEYS if k not in W]
    if absent:
        raise KeyError(f"STOP — {label} weight file missing keys {absent}. Present: {list(W.keys())}.")
print("\nBoth weight payloads contain the required per-node vectors + attention edges.")
print("node_features_used (from Step 1-3):", W_COLOC.get("node_features_used"))


INSPECT: GAT weight files (colocation + random control)

--- gat_weights_colocation :: /kaggle/input/notebooks/energyclimaterp/gat-colocation-gnn/gat_weights_colocation.pt
keys: ['site_id', 'node_features_used', 'embeddings', 'site_attention_received', 'attention_edge_index', 'attention_edge_weight', 'final_reconstruction_mse', 'emb_dim', 'epochs', 'seed', 'note']
   site_id                    tensor shape=(4887,) dtype=torch.int64
   node_features_used         list len=4 sample=['water_stress_bws', 'temperature_c', 'electricity_gwh', 'co2_intensity']
   embeddings                 tensor shape=(4887, 8) dtype=torch.float32
   site_attention_received    tensor shape=(4887,) dtype=torch.float32
   attention_edge_index       tensor shape=(2, 71161) dtype=torch.int64
   attention_edge_weight      tensor shape=(71161,) dtype=torch.float32
   final_reconstruction_mse   float = 0.032908741384744644
   emb_dim                    int = 8
   epochs                     int = 400
   seed         

## Section 2 — Which resources are actually forecastable? 

In [33]:
section("Resource availability check")
have_ws   = "water_stress_bws" in aligned.columns
have_temp = "temperature_c" in aligned.columns and aligned["temperature_c"].notna().any()
have_elec = "electricity_gwh" in aligned.columns
have_co2  = "co2_intensity" in aligned.columns
have_pfaf = "pfaf_id" in aligned.columns

print(f"water stress column present : {have_ws}  -> STATIC per basin (Aqueduct climatology, no time axis) "
      f"=> still NOT forecastable as a series")
print(f"temperature column present  : {have_temp} -> live Open-Meteo per-site SNAPSHOT (Step 1-3) "
      f"=> still NOT forecastable (single reading, no monthly history)")
print(f"electricity present         : {have_elec} -> monthly EMBER series per grid zone     => forecastable")
print(f"carbon (co2_intensity)      : {have_co2}  -> monthly EMBER series per grid zone     => forecastable")
print(f"water (pfaf_id -> G3P tws)  : {have_pfaf} -> monthly G3P Total Water Storage anomaly "
      f"per basin => forecastable (NEW in this version)")

NON_FORECASTABLE = {
    "water_stress": "static per basin (Aqueduct bws_score has no time axis)",
    "temperature":  ("live Open-Meteo per-site SNAPSHOT (a 4th GAT node feature in Step 1-3), "
                     "but a single current reading with NO monthly history => cannot be forecast as a "
                     "series; it influences Step 4 INDIRECTLY through the GAT weights"),
}
print("\nForecastable resources (this run):", list(RESOURCES))
print("Reported non-forecastable:", {k: (v[:70]+"..." if len(v) > 73 else v) for k, v in NON_FORECASTABLE.items()})

if isinstance(W_COLOC.get("node_features_used"), (list, tuple)):
    has_t = any("temp" in str(f).lower() for f in W_COLOC["node_features_used"])
    print(f"\nGAT node_features_used = {W_COLOC['node_features_used']} | temperature among them: {has_t}")


Resource availability check
water stress column present : True  -> STATIC per basin (Aqueduct climatology, no time axis) => still NOT forecastable as a series
temperature column present  : True -> live Open-Meteo per-site SNAPSHOT (Step 1-3) => still NOT forecastable (single reading, no monthly history)
electricity present         : True -> monthly EMBER series per grid zone     => forecastable
carbon (co2_intensity)      : True  -> monthly EMBER series per grid zone     => forecastable
water (pfaf_id -> G3P tws)  : True -> monthly G3P Total Water Storage anomaly per basin => forecastable (NEW in this version)

Forecastable resources (this run): ['electricity', 'carbon', 'water']
Reported non-forecastable: {'water_stress': 'static per basin (Aqueduct bws_score has no time axis)', 'temperature': 'live Open-Meteo per-site SNAPSHOT (a 4th GAT node feature in Step 1-3)...'}

GAT node_features_used = ['water_stress_bws', 'temperature_c', 'electricity_gwh', 'co2_intensity'] | temperature am

## Section 3 

In [34]:
section("3a — Load G3P, assign each site to its NEAREST river basin, build region-id maps")

tws_df = pd.read_csv(paths["g3p_tws"], parse_dates=["time [yyyy-mm-dd]"])
tws_df = tws_df.rename(columns={"time [yyyy-mm-dd]": "date"}).set_index("date").sort_index()
G3P_BASINS = [c.replace(" [mm]", "") for c in tws_df.columns
              if c.endswith("[mm]") and not c.startswith("uncertainty") and c != "global [mm]"]


_cand = [b for b in G3P_BASINS if b in BASIN_CENTROIDS]
_clat = np.array([BASIN_CENTROIDS[b][0] for b in _cand], float)
_clon = np.array([BASIN_CENTROIDS[b][1] for b in _cand], float)
_missing = [b for b in G3P_BASINS if b not in BASIN_CENTROIDS]
print(f"G3P basins: {len(G3P_BASINS)} | with a centroid (candidates): {len(_cand)}"
      + (f" | NO centroid (excluded): {_missing}" if _missing else " | all have centroids"))

def assign_nearest_basin(lat, lon):
    if pd.isna(lat) or pd.isna(lon):
        return "Unknown"
    d = _haversine_km(float(lat), float(lon), _clat, _clon)
    j = int(np.argmin(d))
    return _cand[j] if d[j] <= BASIN_MAX_KM else "Unknown"

aligned["basin_name"] = [assign_nearest_basin(la, lo)
                         for la, lo in zip(aligned["latitude"], aligned["longitude"])]
zone_of_site  = dict(zip(aligned["site_id"].astype(int), aligned["grid_zone_id"]))
basin_of_site = dict(zip(aligned["site_id"].astype(int), aligned["basin_name"]))
REGION_OF = {"electricity": zone_of_site, "carbon": zone_of_site, "water": basin_of_site}

_n = aligned[aligned["is_node"]] if "is_node" in aligned.columns else aligned
print("sites with a resolved grid zone :", sum(1 for v in zone_of_site.values() if isinstance(v, str)))
print(f"nodes assigned to a basin       : {(_n['basin_name']!='Unknown').sum()}/{len(_n)} "
      f"({100*(_n['basin_name']!='Unknown').mean():.1f}%)")
print("geographic sanity — country -> most-common assigned basin (nodes):")
_t = _n[_n.basin_name != "Unknown"].groupby("country")["basin_name"].agg(lambda s: s.value_counts().index[0])
for c in _n["country"].value_counts().head(10).index:
    if c in _t.index:
        print(f"  {c:24s} -> {_t[c]}")
print("basin_name value counts over nodes (top 10):")
print(_n["basin_name"].value_counts().head(10).to_string())


3a — Load G3P, assign each site to its NEAREST river basin, build region-id maps
G3P basins: 100 | with a centroid (candidates): 100 | all have centroids
sites with a resolved grid zone : 4893
nodes assigned to a basin       : 4887/4887 (100.0%)
geographic sanity — country -> most-common assigned basin (nodes):
  United States            -> St.Lawrence
  Netherlands              -> Rhine
  United Kingdom           -> Loire
  Germany                  -> Rhine
  France                   -> Loire
  India                    -> Krishna
  Switzerland              -> Rhine
  Italy                    -> Rhine
  Spain                    -> Loire
  Sweden                   -> Neva
basin_name value counts over nodes (top 10):
basin_name
St.Lawrence                       958
Mississippi River                 862
Rhine                             748
Loire                             548
Colorado River (Pacific Ocean)    456
Brazos River                      397
Columbia River                    1

In [35]:
def _norm(s):
    return (str(s).strip().lower().replace("&", "and").replace(".", "").replace(",", "").replace("  ", " "))

def load_ember_region(path, region):
    cols = pd.read_csv(path, nrows=0).columns.tolist()
    usecols = [c for c in ["State","State code","Area","ISO 3 code","Area type",
                           "Date","Category","Variable","Unit","Value"] if c in cols]
    df = pd.read_csv(path, usecols=usecols); df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    gen = df[(df["Category"]=="Electricity generation") & (df["Variable"].str.lower()=="total generation")
             & (df["Unit"].isin(["GWh","TWh"]))].copy()
    gen["electricity_gwh"] = np.where(gen["Unit"]=="TWh", gen["Value"]*1000.0, gen["Value"])
    car = df[(df["Category"]=="Power sector emissions") & (df["Variable"]=="CO2 intensity")].copy()
    car["co2_intensity"] = car["Value"]
    if region in ("US","India"):
        iso = "USA" if region=="US" else "IND"
        for g in (gen, car):
            g["zone_id"] = iso + "|" + g["State"].map(_norm)
            g.drop(g.index[g["State"].astype(str).str.lower().str.contains("total")], inplace=True)
    else:
        gen = gen[gen["Area type"]=="Country or economy"]; car = car[car["Area type"]=="Country or economy"]
        gen["zone_id"] = gen["ISO 3 code"].astype(str); car["zone_id"] = car["ISO 3 code"].astype(str)
    gen = gen.dropna(subset=["Value","zone_id","Date"]); car = car.dropna(subset=["Value","zone_id","Date"])
    g = gen.groupby(["zone_id","Date"], as_index=False)["electricity_gwh"].mean()
    c = car.groupby(["zone_id","Date"], as_index=False)["co2_intensity"].mean()
    return pd.merge(g, c, on=["zone_id","Date"], how="outer")

section("3b — EMBER zone panel (electricity + carbon)")
ember = pd.concat([load_ember_region(paths["ember_us"], "US"),
                   load_ember_region(paths["ember_eu"], "EU"),
                   load_ember_region(paths["ember_in"], "India")], ignore_index=True)

zones_with_nodes = {z for z in pd.unique(aligned.loc[aligned["is_node"], "grid_zone_id"]) if isinstance(z, str)} \
    if "is_node" in aligned.columns else {z for z in aligned["grid_zone_id"].dropna().unique()}
print("zones containing >=1 graph node:", len(zones_with_nodes))

region_series = {"electricity": {}, "carbon": {}, "water": {}}
for z in sorted(zones_with_nodes):
    sub = ember[ember["zone_id"] == z].set_index("Date").sort_index()
    if sub.empty: continue
    for rk, col in [("electricity","electricity_gwh"), ("carbon","co2_intensity")]:
        s = sub[col].dropna(); s = s[~s.index.duplicated(keep="first")]
        if len(s) >= MIN_ZONE_MONTHS:
            try: s.index.freq = "MS"
            except Exception: pass
            region_series[rk][z] = s
print("zones with usable electricity series:", len(region_series["electricity"]))
print("zones with usable carbon series     :", len(region_series["carbon"]))


3b — EMBER zone panel (electricity + carbon)
zones containing >=1 graph node: 94
zones with usable electricity series: 94
zones with usable carbon series     : 94


In [36]:
section("3c — G3P water-storage series per basin (from the nearest-basin assignment in 3a)")
print(f"G3P tws file: {len(G3P_BASINS)} basin columns, {len(tws_df)} monthly rows "
      f"({tws_df.index.min().date()}..{tws_df.index.max().date()})")

basins_with_nodes = {b for b in pd.unique(aligned.loc[aligned["is_node"], "basin_name"]) if b != "Unknown"} \
    if "is_node" in aligned.columns else {b for b in aligned["basin_name"].unique() if b != "Unknown"}
print("basins containing >=1 graph node:", len(basins_with_nodes), "->", sorted(basins_with_nodes))

for name in G3P_BASINS:
    if name not in basins_with_nodes:
        continue
    s = tws_df[f"{name} [mm]"].dropna(); s = s[~s.index.duplicated(keep="first")]
    if len(s) >= MIN_ZONE_MONTHS:
        try: s.index.freq = "MS"
        except Exception: pass
        region_series["water"][name] = s
print("basins with usable water (tws) series:", len(region_series["water"]))
if not region_series["water"]:
    print("!! No basin matched between the graph nodes and the G3P columns — water will be reported "
          "as skipped in Section 6/8, not fabricated.")


3c — G3P water-storage series per basin (from the nearest-basin assignment in 3a)
G3P tws file: 100 basin columns, 225 monthly rows (2002-04-16..2023-09-16)
basins containing >=1 graph node: 24 -> ['Albany River', 'Bravo', 'Brazos River', 'Colorado River (Pacific Ocean)', 'Columbia River', 'Danube', 'Dniepr', 'Elbe River', 'Ganges', 'Godavari', 'Grisalva', 'Indus', 'Krishna', 'Loire', 'Mahanadi River (Mahahadi)', 'Mississippi River', 'Nelson River', 'Neva', 'Oder River', 'Rhine', 'Senegal', 'St.Lawrence', 'Tigris & Euphrates', 'Wisla']
basins with usable water (tws) series: 24


## Section 4 — GAT weight structures (colocation + random control)


In [37]:
section("Weight structures (zone + basin per site)")

def build_weight_struct(W, label):
    site_id = W["site_id"].cpu().numpy().astype(int)
    att = W["site_attention_received"].cpu().numpy().astype(float)
    ei = W["attention_edge_index"].cpu().numpy()
    ew = W["attention_edge_weight"].cpu().numpy().astype(float)
    if ew.ndim > 1: ew = ew.mean(axis=1)
    keep = ei[0] != ei[1]
    ei = ei[:, keep]; ew = ew[keep]
    rng = att.max() - att.min()
    a_site = ALPHA_MAX * ((att - att.min()) / rng if rng > 0 else np.zeros_like(att))
    zones  = np.array([zone_of_site.get(int(s), None)  for s in site_id], dtype=object)
    basins = np.array([basin_of_site.get(int(s), None) for s in site_id], dtype=object)
    n_nbr = np.bincount(ei[1], minlength=len(site_id))
    print(f"{label}: N={len(site_id)} nodes | edges(no self-loop)={ei.shape[1]} | "
          f"att min/mean/max={att.min():.4f}/{att.mean():.4f}/{att.max():.4f} | "
          f"a_site min/mean/max={a_site.min():.3f}/{a_site.mean():.3f}/{a_site.max():.3f} | "
          f"nodes with >=1 neighbour={(n_nbr>0).sum()}")
    return {"site_id": site_id, "zones": zones, "basins": basins, "a": a_site,
            "src": ei[0], "dst": ei[1], "w": ew}

S_COLOC  = build_weight_struct(W_COLOC,  "colocation")
S_RANDOM = build_weight_struct(W_RANDOM, "random_control")

REGION_KEY = {"electricity": "zones", "carbon": "zones", "water": "basins"}  


Weight structures (zone + basin per site)
colocation: N=4887 nodes | edges(no self-loop)=66274 | att min/mean/max=0.0099/0.0823/1.0000 | a_site min/mean/max=0.000/0.037/0.500 | nodes with >=1 neighbour=4882
random_control: N=4887 nodes | edges(no self-loop)=66274 | att min/mean/max=0.0323/0.0737/0.2500 | a_site min/mean/max=0.000/0.095/0.500 | nodes with >=1 neighbour=4887


## Section 5 — Native-injection demo 

In [38]:
section("Native-injection demo (SARIMAX exog) — electricity, zone level")

def compute_rmse(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    return np.sqrt(np.mean((y_true - y_pred) ** 2))

def compute_mape(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true, float), np.asarray(y_pred, float)
    mask = np.abs(y_true) > 1e-8
    return float(np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100) if mask.any() else np.nan

def fit_sarima(train_series, test_len, order=(1,1,1), seasonal_order=(1,1,0,12), exog=None, exog_future=None):
    model = SARIMAX(train_series, order=order, seasonal_order=seasonal_order,
                    exog=exog, enforce_stationarity=False, enforce_invertibility=False)
    result = model.fit(disp=False, maxiter=200)
    forecast = result.get_forecast(test_len, exog=exog_future).predicted_mean.values
    return forecast, result

def region_neighbour_series(region_id, resource, struct):
    key = REGION_KEY[resource]
    ids = struct[key]
    r_nodes = np.where(ids == region_id)[0]
    if len(r_nodes) == 0: return None
    nbr_w = {}; rset = set(r_nodes)
    for k in range(struct["src"].shape[0]):
        if struct["dst"][k] in rset:
            nr = ids[struct["src"][k]]
            if isinstance(nr, str) and nr != region_id and nr in region_series[resource]:
                nbr_w[nr] = nbr_w.get(nr, 0.0) + struct["w"][k]
    if not nbr_w: return None
    tot = sum(nbr_w.values())
    idx = region_series[resource][region_id].index
    acc = pd.Series(0.0, index=idx)
    for nr, w in nbr_w.items():
        acc = acc.add((w / tot) * region_series[resource][nr].reindex(idx).ffill().bfill(), fill_value=0.0)
    return acc

demo_log = []
demo_zones = [z for z in region_series["electricity"]
              if region_neighbour_series(z, "electricity", S_COLOC) is not None][:2]
if not demo_zones:
    print("No zone has cross-zone co-location neighbours for 'electricity' -> native demo not applicable here.")
for z in demo_zones:
    r = "electricity"; s = region_series[r][z]; train, test = s.iloc[:-HOLDOUT], s.iloc[-HOLDOUT:]
    nbr = region_neighbour_series(z, r, S_COLOC)
    try:
        base_fc, _ = fit_sarima(train, len(test))
        ex_tr = nbr.reindex(train.index).ffill().bfill().values.reshape(-1, 1)
        ex_fu = nbr.reindex(test.index).ffill().bfill().values.reshape(-1, 1)
        exog_fc, _ = fit_sarima(train, len(test), exog=ex_tr, exog_future=ex_fu)
        d = float(np.mean(np.abs(exog_fc - base_fc)))
        print(f"[SARIMAX] zone {z}: mean|exog-base| = {d:.3f}  -> {'CHANGED' if d>1e-6 else 'no change'}")
        demo_log.append({"region": z, "resource": r, "model": "SARIMAX_exog", "mean_abs_delta": d})
    except Exception as e:
        print(f"[SARIMAX] zone {z}: demo failed ({type(e).__name__}: {e})")
print("\nNative-injection demo confirms the weight CAN be wired natively (deltas above). "
      "The scalable per-site deliverable uses the blend (Section 8).")


Native-injection demo (SARIMAX exog) — electricity, zone level
[SARIMAX] zone BEL: mean|exog-base| = 80.455  -> CHANGED
[SARIMAX] zone CHE: mean|exog-base| = 187.154  -> CHANGED

Native-injection demo confirms the weight CAN be wired natively (deltas above). The scalable per-site deliverable uses the blend (Section 8).


## Section 6 — PASS 1: base (unweighted) forecasts

In [39]:
base = {}            # base[(model, resource)][region_id] = (dates, values)
region_dates = {}    # region_dates[(region_id, resource)] = holdout date index
fail_log = []        # (region_id, resource, model, reason)

def _record(model, resource, region_id, test_index, forecast_vals):
    base.setdefault((model, resource), {})[region_id] = (test_index.values, np.asarray(forecast_vals))
    region_dates[(region_id, resource)] = test_index

def _quick_rmse_so_far(model):
    '''Actual-vs-forecast RMSE for whatever this model has produced so far — printed after every
    model finishes, so results are visible immediately rather than only at the very end.'''
    rows = []
    for resource in RESOURCES:
        zd = base.get((model, resource), {})
        for region_id, (dts, vals) in zd.items():
            actual = region_series[resource][region_id].reindex(pd.to_datetime(dts)).values
            m = ~np.isnan(actual)
            if m.sum():
                rows.append((resource, compute_rmse(actual[m], np.asarray(vals)[m])))
    if not rows:
        print(f"  ({model}: no series scored yet)")
        return
    d = pd.DataFrame(rows, columns=["resource", "rmse"])
    print(f"  {model} quick RMSE by resource (unweighted, this model only):")
    print("   " + d.groupby("resource")["rmse"].mean().round(3).to_string().replace("\n", "\n   "))

print("base / region_dates / fail_log initialised.")

base / region_dates / fail_log initialised.


In [40]:
section("6A — SARIMA fastest model")
for resource, region_type in RESOURCES.items():
    series_dict = region_series[resource]
    for region_id, s in series_dict.items():
        train, test = s.iloc[:-HOLDOUT], s.iloc[-HOLDOUT:]
        try:
            fc, _ = fit_sarima(train, len(test))
            _record("SARIMA", resource, region_id, test.index, fc)
        except Exception as e:
            fail_log.append((region_id, resource, "SARIMA", f"{type(e).__name__}: {e}"[:150]))
print(f"SARIMA done: {sum(len(v) for k,v in base.items() if k[0]=='SARIMA')} region-series forecast, "
      f"{sum(1 for f in fail_log if f[2]=='SARIMA')} failed.")
_quick_rmse_so_far("SARIMA")


6A — SARIMA fastest model
SARIMA done: 212 region-series forecast, 0 failed.
  SARIMA quick RMSE by resource (unweighted, this model only):
   resource
   carbon          38.291
   electricity    612.002
   water           45.444


## THIS IS THE NEXUS CODE 

add API IN PLACE OF KAGGLE SCERETS

In [ ]:
import time as _time
section("6B — Nexus (MINIMISED OpenAI: exactly 1 call per region-series; no SARIMA fallback)")


NEXUS_MAX_LLM_CALLS = 200     # <-- lower this to spend fewer calls; None = 1 call/series, no cap
NEXUS_SLEEP         = 1.2      # seconds between calls (stay under requests-per-minute limits)
NEXUS_RETRIES       = 3        # attempts per series on 429/quota before falling back
NEXUS_BACKOFF       = 8.0      # base backoff seconds (x attempt) on 429
NEXUS_HIST_POINTS   = 48       # only send the last N months in the prompt (smaller = cheaper/faster)


def _nexus_naive_fallback(series_train, pred_len):
    mm = series_train.groupby(series_train.index.month).mean()
    last_val = float(series_train.iloc[-1])
    last = series_train.index[-1]
    fc = np.array([
        0.5 * last_val + 0.5 * mm.get(((last.month - 1 + i + 1) % 12) + 1, series_train.mean())
        for i in range(pred_len)
    ], float)
    return fc

# ==================== PUT YOUR OPENAI API KEY HERE ====================
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")   # <-- paste your key between the quotes
# ========================================================================

NEXUS_LLM_OK = False; openai_client = None; OPENAI_MODEL = "gpt-4o-mini"
OPENAI_CALLS = 0
try:
    from openai import OpenAI
    if not OPENAI_API_KEY or OPENAI_API_KEY.startswith("sk-REPLACE"):
        raise ValueError("OPENAI_API_KEY not set — paste your real key above")
    openai_client = OpenAI(api_key=OPENAI_API_KEY)
    NEXUS_LLM_OK = True
    print(f"Nexus LLM ready | model={OPENAI_MODEL} | budget: <= {NEXUS_MAX_LLM_CALLS} calls, "
          f"1 per series, {NEXUS_SLEEP}s apart")
except Exception as e:
    print("=" * 78)
    print("!!! WARNING: NEXUS LLM IS NOT WORKING — RUNNING ON NAIVE FALLBACK ONLY !!!")
    print(f"Reason: {type(e).__name__}: {e}")
    print("Every forecast from this stage will be a naive seasonal estimate,")
    print("NOT an LLM forecast. Fix the API key above and rerun this cell.")
    print("=" * 78)

def _nexus_llm_once(series_train, pred_len):
    H = [{"m": str(d)[:7], "v": round(float(v), 3)} for d, v in series_train.items()][-NEXUS_HIST_POINTS:]
    prompt = ("You are an expert monthly time-series forecaster that internally combines macro-trend, "
              "12-month seasonality, and bias-calibration reasoning in a SINGLE pass. "
              f"History (last {len(H)} months): {json.dumps(H)}. "
              f"Forecast the next {pred_len} monthly values. "
              f"Return ONLY a JSON array of exactly {pred_len} numbers — no prose, no keys.")
    resp = openai_client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[{"role": "user", "content": prompt}],
    )
    txt = resp.choices[0].message.content.strip().strip("`")
    txt = txt[txt.find("["): txt.rfind("]") + 1] if "[" in txt else txt
    arr = np.asarray(json.loads(txt), float)[:pred_len]
    if len(arr) != pred_len or not np.isfinite(arr).all():
        raise ValueError("LLM returned malformed / wrong-length forecast")
    return arr

def nexus_forecast(series_train, pred_len):
    global OPENAI_CALLS
    if NEXUS_LLM_OK and (NEXUS_MAX_LLM_CALLS is None or OPENAI_CALLS < NEXUS_MAX_LLM_CALLS):
        for attempt in range(NEXUS_RETRIES):
            try:
                OPENAI_CALLS += 1
                fc = _nexus_llm_once(series_train, pred_len)
                if NEXUS_SLEEP > 0: _time.sleep(NEXUS_SLEEP)
                return fc, "LLM"
            except Exception as e:
                msg = str(e).lower()
                if any(t in msg for t in ["429", "quota", "rate", "resource_exhausted", "rate_limit"]) and attempt < NEXUS_RETRIES - 1:
                    _time.sleep(NEXUS_BACKOFF * (attempt + 1)); continue
                break  # non-retryable, or out of retries -> naive fallback
    return _nexus_naive_fallback(series_train, pred_len), "naive-fallback"





# --- NEXUS LOCAL BACKEND (begin) ---
import sys as _sys
_sys.path.insert(0, os.path.join("..", "src"))
from nexus_local import make_nexus_forecast
NEXUS_BACKEND = "chronos"
_nexus_impl = make_nexus_forecast(NEXUS_BACKEND)
NEXUS_LLM_OK = True
OPENAI_CALLS = 0

def nexus_forecast(series_train, pred_len):
    """Wraps the local backend in the notebook's own bookkeeping contract.
    §6B counts into nexus_src{"LLM", "naive-fallback"} and warns when
    OPENAI_CALLS == 0, so a successful model call must report as "LLM"
    (meaning: the Nexus model produced this, not the fallback)."""
    global OPENAI_CALLS
    fc, tag = _nexus_impl(series_train, pred_len)
    if tag != "naive-fallback":
        OPENAI_CALLS += 1
        return fc, "LLM"
    return fc, "naive-fallback"

print(f"Nexus backend: {NEXUS_BACKEND} (local, no API key, deterministic) "
      f"-- counted under the 'LLM' key in this section's tallies")
# --- NEXUS LOCAL BACKEND (end) ---

n_series_total = sum(len(region_series[r]) for r in RESOURCES)
print(f"Nexus running over {n_series_total} region-series — at most 1 OpenAI call each "
      f"(hard-capped at {NEXUS_MAX_LLM_CALLS}). Fallback is naive seasonal, so quota exhaustion is safe.")
nexus_src = {"LLM": 0, "naive-fallback": 0}
for resource, region_type in RESOURCES.items():
    for region_id, s in region_series[resource].items():
        train, test = s.iloc[:-HOLDOUT], s.iloc[-HOLDOUT:]
        try:
            fc, src = nexus_forecast(train, len(test))
            nexus_src[src] += 1
            _record("Nexus", resource, region_id, test.index, fc)
        except Exception as e:
            fail_log.append((region_id, resource, "Nexus", f"{type(e).__name__}: {e}"[:150]))

print(f"Nexus done: {sum(len(v) for k,v in base.items() if k[0]=='Nexus')} region-series forecast, "
      f"{sum(1 for f in fail_log if f[2]=='Nexus')} failed.")
print(f"  OpenAI calls actually made: {OPENAI_CALLS} | via LLM: {nexus_src['LLM']} | "
      f"via naive fallback: {nexus_src['naive-fallback']}")

if OPENAI_CALLS == 0 or nexus_src["LLM"] == 0:
    print("=" * 78)
    print("!!! WARNING: NEXUS RAN WITH ZERO SUCCESSFUL LLM CALLS !!!")
    print("All results for this stage are naive-fallback, not real LLM forecasts.")
    print("Check your API key, quota, and network before trusting these numbers.")
    print("=" * 78)
elif nexus_src["naive-fallback"] > 0:
    print(f"NOTE: {nexus_src['naive-fallback']} of {n_series_total} series fell back to naive "
          f"(quota/errors) — LLM succeeded for the rest.")

_quick_rmse_so_far("Nexus")


6B — Nexus (MINIMISED Gemini: exactly 1 call per region-series; graceful stat-fallback)
!! Nexus LLM unavailable (BackendError: Unexpected response from the service. Response: {'errors': ['No user secrets exist for kernel id 131584474 and label CCAI_NEXUS.'], 'error': {'code': 5}, 'wasSuccessful': False}.) -> statistical fallback for ALL series (0 Gemini calls). The notebook still runs and does NOT fabricate.
Nexus running over 212 region-series — at most 1 Gemini call each (hard-capped at 200). Fallback is statistical, so quota exhaustion is safe.
Nexus done: 212 region-series forecast, 0 failed.
  Gemini calls actually made: 0 | via LLM: 0 | via statistical fallback: 212
  Nexus quick RMSE by resource (unweighted, this model only):
   resource
   carbon          36.213
   electricity    581.237
   water           35.873


## Section 7 — Sanity check: coverage across both models × 3 resources



In [42]:
section("Coverage — region-series forecast per model x resource")
cov_rows = []
for m in MODELS:
    for r in RESOURCES:
        n_ok = len(base.get((m, r), {}))
        n_total = len(region_series[r])
        n_fail = sum(1 for f in fail_log if f[1] == r and f[2] == m)
        cov_rows.append({"model": m, "resource": r, "region_series_total": n_total,
                         "forecast_ok": n_ok, "failed_or_skipped": n_fail})
cov_df = pd.DataFrame(cov_rows)
print(cov_df.to_string(index=False))
if fail_log:
    print(f"\n{len(fail_log)} total failures/skips logged (see forecast_failures_log.csv in Section 9). "
          f"First 5:")
    for row in fail_log[:5]:
        print("  ", row)


Coverage — region-series forecast per model x resource
 model    resource  region_series_total  forecast_ok  failed_or_skipped
SARIMA electricity                   94           94                  0
SARIMA      carbon                   94           94                  0
SARIMA       water                   24           24                  0
 Nexus electricity                   94           94                  0
 Nexus      carbon                   94           94                  0
 Nexus       water                   24           24                  0


## Section 8 — PASS 2: per-site weighted forecasts (GAT + random control)

In [43]:
section("Pass 2 -- per-site weighted forecasts (zone- and basin-based resources)")

def region_base_matrix(region_ids, bz, H=HOLDOUT):
    N = len(region_ids)
    node_base = np.full((N, H), np.nan); node_dates = np.empty(N, dtype=object)
    for i in range(N):
        rid = region_ids[i]
        if isinstance(rid, str) and rid in bz:
            dts, vals = bz[rid]
            if len(vals) == H:
                node_base[i] = vals; node_dates[i] = dts
    return node_base, node_dates, ~np.isnan(node_base).any(axis=1)

def blend_nodes(node_base, valid_node, src, dst, w, a_vec):
    N, H = node_base.shape
    num = np.zeros((N, H)); den = np.zeros(N)
    good = valid_node[src]
    se, de, we = src[good], dst[good], w[good]
    np.add.at(num, de, we[:, None] * node_base[se]); np.add.at(den, de, we)
    nbr = np.where(den[:, None] > 0, num / np.where(den[:, None] > 0, den[:, None], 1.0), node_base)
    a_eff = np.where(den > 0, a_vec, 0.0)
    return (1 - a_eff)[:, None] * node_base + a_eff[:, None] * nbr, (den > 0)

def weighted_forecasts(struct, label):
    out_rows = {"unweighted": [], label: []}
    changed_counter = {"total": 0, "changed": 0}
    site_id = struct["site_id"]; a = struct["a"]; src, dst, w = struct["src"], struct["dst"], struct["w"]
    for m in MODELS:
        for r in RESOURCES:
            bz = base.get((m, r), {})
            if not bz:
                continue
            region_ids = struct[REGION_KEY[r]]
            node_base, node_dates, valid_node = region_base_matrix(region_ids, bz)
            weighted, _ = blend_nodes(node_base, valid_node, src, dst, w, a)
            for i in np.where(valid_node)[0]:
                dts = node_dates[i]
                for h in range(HOLDOUT):
                    out_rows["unweighted"].append((int(site_id[i]), r, m, pd.Timestamp(dts[h]), float(node_base[i, h])))
                    out_rows[label].append((int(site_id[i]), r, m, pd.Timestamp(dts[h]), float(weighted[i, h])))
                changed_counter["total"] += 1
                if np.mean(np.abs(weighted[i] - node_base[i])) > 1e-9:
                    changed_counter["changed"] += 1
    cols = ["site_id", "resource", "model", "timestamp", "value"]
    dfs = {k: pd.DataFrame(v, columns=cols) for k, v in out_rows.items()}
    return dfs, changed_counter

df_coloc, chg_coloc = weighted_forecasts(S_COLOC, "gat_weighted")
df_random, chg_random = weighted_forecasts(S_RANDOM, "random_weighted")

df_unweighted = df_coloc["unweighted"]
df_gat = df_coloc["gat_weighted"]
df_rnd = df_random["random_weighted"]

print("Sanity — site x resource x model combinations emitted:")
print(f"  unweighted rows      : {len(df_unweighted)}")
print(f"  gat_weighted rows    : {len(df_gat)}  | changed vs unweighted: "
      f"{chg_coloc['changed']}/{chg_coloc['total']} site-series "
      f"({100*chg_coloc['changed']/max(chg_coloc['total'],1):.1f}%)")
print(f"  random_weighted rows : {len(df_rnd)}  | changed vs unweighted: "
      f"{chg_random['changed']}/{chg_random['total']} site-series "
      f"({100*chg_random['changed']/max(chg_random['total'],1):.1f}%)")
print("\nBy resource — rows emitted:")
print(df_unweighted.groupby("resource").size().to_string())
print("\nNOTE: site-series that did NOT change had only same-zone/same-basin (or no) co-location "
      "neighbours, so the neighbour signal equals the site's own base. Reported, not hidden.")


Pass 2 -- per-site weighted forecasts (zone- and basin-based resources)
Sanity — site x resource x model combinations emitted:
  unweighted rows      : 351864
  gat_weighted rows    : 351864  | changed vs unweighted: 2650/29322 site-series (9.0%)
  random_weighted rows : 351864  | changed vs unweighted: 29316/29322 site-series (100.0%)

By resource — rows emitted:
resource
carbon         117288
electricity    117288
water          117288

NOTE: site-series that did NOT change had only same-zone/same-basin (or no) co-location neighbours, so the neighbour signal equals the site's own base. Reported, not hidden.


## Section 9 —  coverage summary

In [44]:
section("Save outputs")
f_un = os.path.join(WORK, "forecasts_unweighted.csv")
f_ga = os.path.join(WORK, "forecasts_gat_weighted.csv")
f_rc = os.path.join(WORK, "forecasts_random_control_weighted.csv")
df_unweighted.to_csv(f_un, index=False)
df_gat.to_csv(f_ga, index=False)
df_rnd.to_csv(f_rc, index=False)

base_rows = []
for (m, r), zd in base.items():
    for region_id, (dts, vals) in zd.items():
        for h in range(len(vals)):
            base_rows.append((region_id, r, m, pd.Timestamp(dts[h]), float(vals[h])))
pd.DataFrame(base_rows, columns=["region_id","resource","model","timestamp","value"]).to_csv(
    os.path.join(WORK, "per_region_base_forecasts.csv"), index=False)
pd.DataFrame(fail_log, columns=["region_id","resource","model","reason"]).to_csv(
    os.path.join(WORK, "forecast_failures_log.csv"), index=False)

injection_log = {
    "forecast_unit": {"electricity": "grid_zone", "carbon": "grid_zone",
                      "water": f"basin (nearest G3P river-basin centroid, cutoff {BASIN_MAX_KM} km)"},
    "resources_forecast": list(RESOURCES),
    "resources_skipped": NON_FORECASTABLE,
    "models": MODELS,
    "model_run_order_rationale": "SARIMA baseline, then Nexus (live LLM calls)",
    "weight_field_used": "site_attention_received (min-max scaled, x ALPHA_MAX)",
    "ALPHA_MAX": ALPHA_MAX,
    "native_injection_demo": demo_log,
    "changed_fraction_gat": chg_coloc["changed"] / max(chg_coloc["total"], 1),
    "changed_fraction_random": chg_random["changed"] / max(chg_random["total"], 1),
}
with open(os.path.join(WORK, "weight_injection_log.json"), "w") as fh:
    json.dump(injection_log, fh, indent=2, default=str)

def n_combos(df): return df.groupby(["site_id","resource"]).ngroups if len(df) else 0
print("COVERAGE SUMMARY")
print(f"  site x resource combos with a forecast under ALL 3 conditions: {n_combos(df_unweighted)}")
print(f"  (unweighted={n_combos(df_unweighted)}, gat={n_combos(df_gat)}, random={n_combos(df_rnd)})")
print(f"  per-model x resource regions forecast:")
for m in MODELS:
    for r in RESOURCES:
        print(f"     {m:8s} {r:11s}: {len(base.get((m,r), {}))} regions")
print(f"  failures/skips: {len(fail_log)} (see forecast_failures_log.csv)")
print("\nFiles written to /kaggle/working/:")
for p in [f_un, f_ga, f_rc, "per_region_base_forecasts.csv", "forecast_failures_log.csv", "weight_injection_log.json"]:
    pp = p if os.path.isabs(p) else os.path.join(WORK, p)
    print(f"   {pp}  ({os.path.getsize(pp)/1024:.1f} KiB)")


Save outputs
COVERAGE SUMMARY
  site x resource combos with a forecast under ALL 3 conditions: 14661
  (unweighted=14661, gat=14661, random=14661)
  per-model x resource regions forecast:
     SARIMA   electricity: 94 regions
     SARIMA   carbon     : 94 regions
     SARIMA   water      : 24 regions
     Nexus    electricity: 94 regions
     Nexus    carbon     : 94 regions
     Nexus    water      : 24 regions
  failures/skips: 0 (see forecast_failures_log.csv)

Files written to /kaggle/working/:
   /kaggle/working/forecasts_unweighted.csv  (16962.6 KiB)
   /kaggle/working/forecasts_gat_weighted.csv  (16960.5 KiB)
   /kaggle/working/forecasts_random_control_weighted.csv  (16959.8 KiB)
   /kaggle/working/per_region_base_forecasts.csv  (275.1 KiB)
   /kaggle/working/forecast_failures_log.csv  (0.0 KiB)
   /kaggle/working/weight_injection_log.json  (1.2 KiB)


## Section 10 — STEP 5: metric evaluation of the three forecast sets


In [45]:

_ex_r = next(iter(region_series)); _ex_z = next(iter(region_series[_ex_r]))
_s = region_series[_ex_r][_ex_z]
print(f"Split check on example region '{_ex_z}' / {_ex_r}: series {_s.index[0].date()}..{_s.index[-1].date()} "
      f"({len(_s)} months) | TRAIN = first {len(_s)-HOLDOUT} | TEST(ground truth) = last {HOLDOUT} "
      f"({_s.index[-HOLDOUT].date()}..{_s.index[-1].date()})")
assert (region_dates[(_ex_z, _ex_r)] == _s.index[-HOLDOUT:]).all(), "Ground-truth window mismatch vs Section 6 split!"

act_rows = []
for (region_id, r), dtidx in region_dates.items():
    s = region_series[r][region_id]
    for ts in dtidx:
        act_rows.append((region_id, r, pd.Timestamp(ts), float(s.loc[ts])))
actuals = pd.DataFrame(act_rows, columns=["region_id", "resource", "timestamp", "actual"])
print("actuals rows (region x resource x holdout-month):", len(actuals),
      "| resources scored:", sorted(actuals.resource.unique()))
print("NOTE: water_stress / temperature are NOT scored — they have no forecast (static / absent). See Section 2.")

Split check on example region 'AUT' / electricity: series 2015-01-01..2026-04-01 (136 months) | TRAIN = first 124 | TEST(ground truth) = last 12 (2025-05-01..2026-04-01)
actuals rows (region x resource x holdout-month): 2544 | resources scored: ['carbon', 'electricity', 'water']
NOTE: water_stress / temperature are NOT scored — they have no forecast (static / absent). See Section 2.


In [46]:
section("STEP 5 -- score the three forecast sets (ensemble across models)")
def _metrics(g):
    e = g["value"].values - g["actual"].values; a = g["actual"].values; mask = np.abs(a) > 1e-8
    return pd.Series({"MAE": float(np.mean(np.abs(e))), "RMSE": float(np.sqrt(np.mean(e ** 2))),
                      "MAPE": float(np.mean(np.abs(e[mask] / a[mask])) * 100) if mask.any() else np.nan})

def region_id_of(resource, site_id):
    return REGION_OF[resource].get(site_id)

def ensemble_score(df, cond):
    d = df.copy(); d["region_id"] = d.apply(lambda row: region_id_of(row["resource"], int(row["site_id"])), axis=1)
    ens = d.groupby(["site_id", "region_id", "resource", "timestamp"], as_index=False)["value"].mean()
    m = ens.merge(actuals, on=["region_id", "resource", "timestamp"], how="inner")
    met = m.groupby(["site_id", "resource"]).apply(_metrics).reset_index(); met["condition"] = cond
    return met

cond_dfs = {"unweighted": df_unweighted, "gat_weighted": df_gat, "random_control": df_rnd}
met_all = pd.concat([ensemble_score(df, c) for c, df in cond_dfs.items()], ignore_index=True)

wide = met_all.pivot_table(index=["site_id", "resource"], columns="condition", values=["MAPE", "RMSE", "MAE"])
wide.columns = [f"{met}_{cond}" for met, cond in wide.columns]
wide = wide.reset_index()
wide.to_csv(os.path.join(WORK, "step5_metrics_site_resource.csv"), index=False)
print("site x resource scored:", wide[["site_id", "resource"]].drop_duplicates().shape[0])
print("saved step5_metrics_site_resource.csv | columns:", list(wide.columns))
print(wide.head(3).to_string())


STEP 5 -- score the three forecast sets (ensemble across models)
site x resource scored: 14661
saved step5_metrics_site_resource.csv | columns: ['site_id', 'resource', 'MAE_gat_weighted', 'MAE_random_control', 'MAE_unweighted', 'MAPE_gat_weighted', 'MAPE_random_control', 'MAPE_unweighted', 'RMSE_gat_weighted', 'RMSE_random_control', 'RMSE_unweighted']
   site_id     resource  MAE_gat_weighted  MAE_random_control  MAE_unweighted  MAPE_gat_weighted  MAPE_random_control  MAPE_unweighted  RMSE_gat_weighted  RMSE_random_control  RMSE_unweighted
0        0       carbon         12.425481           15.010037       12.425481           8.906947            11.184012         8.906947          14.635863            17.947845        14.635863
1        0  electricity        790.678810          899.806325      790.678810           3.667004             4.138155         3.667004        1037.471803          1165.463342      1037.471803
2        0        water         31.703612           30.710906       3

In [47]:
section("STEP 5 -- aggregate verdict per resource (raw numbers, no spin)")
agg = (met_all.groupby(["resource", "condition"])[["MAPE", "RMSE", "MAE"]].mean().round(4))
print(agg.to_string())
agg.to_csv(os.path.join(WORK, "step5_aggregate_by_resource.csv"))

print("\nFactual comparison (mean over sites) — computed here, not pre-written:")
for r in sorted(met_all.resource.unique()):
    row = agg.xs(r, level="resource")
    for metric in ["MAPE", "RMSE", "MAE"]:
        u = row.loc["unweighted", metric]; g = row.loc["gat_weighted", metric]; rc = row.loc["random_control", metric]
        vs_u = "lower(better)" if g < u else ("higher(worse)" if g > u else "equal")
        vs_r = "lower(better)" if g < rc else ("higher(worse)" if g > rc else "equal")
        print(f"  {r:11s} {metric:5s}: unweighted={u:.4f}  gat={g:.4f} ({vs_u} vs unweighted)  "
              f"random={rc:.4f} (gat {vs_r} vs random)")
print("\n(Verdict is stated numerically only. Significance tested next; a null/negative result is a valid outcome.)")


STEP 5 -- aggregate verdict per resource (raw numbers, no spin)
                                MAPE       RMSE        MAE
resource    condition                                     
carbon      gat_weighted     11.7145    27.7454    23.5705
            random_control   13.6806    29.4550    25.2203
            unweighted       11.7276    27.7490    23.5741
electricity gat_weighted      7.0959  1078.1668   933.6202
            random_control   40.1585  1779.1416  1633.4771
            unweighted        6.7642  1075.7967   931.3596
water       gat_weighted    199.6892    42.1266    35.9090
            random_control  202.5603    41.7146    35.6141
            unweighted      199.7282    42.1466    35.9290

Factual comparison (mean over sites) — computed here, not pre-written:
  carbon      MAPE : unweighted=11.7276  gat=11.7145 (lower(better) vs unweighted)  random=13.6806 (gat lower(better) vs random)
  carbon      RMSE : unweighted=27.7490  gat=27.7454 (lower(better) vs unweighted)  r

In [48]:
section("STEP 5 -- paired significance tests (per-site errors)")
sig_rows = []
piv = met_all.pivot_table(index=["site_id", "resource"], columns="condition", values="RMSE")
for r in sorted(met_all.resource.unique()):
    sub = piv.xs(r, level="resource")
    for other in ["unweighted", "random_control"]:
        pair = sub[["gat_weighted", other]].dropna()
        a, b = pair["gat_weighted"].values, pair[other].values
        n = len(pair); diff = a - b
        if n < 3 or np.allclose(diff, 0):
            sig_rows.append({"resource": r, "comparison": f"gat_vs_{other}", "n": n,
                             "mean_diff_rmse": float(np.mean(diff)) if n else np.nan,
                             "t_stat": np.nan, "t_p": np.nan, "wilcoxon_stat": np.nan, "wilcoxon_p": np.nan,
                             "note": "identical or n<3 -> no test (likely weighted==unweighted for these sites)"})
            continue
        t_stat, t_p = stats.ttest_rel(a, b)
        try: w_stat, w_p = stats.wilcoxon(a, b)
        except Exception: w_stat, w_p = np.nan, np.nan
        sig_rows.append({"resource": r, "comparison": f"gat_vs_{other}", "n": n,
                         "mean_diff_rmse": float(np.mean(diff)), "t_stat": float(t_stat), "t_p": float(t_p),
                         "wilcoxon_stat": float(w_stat) if w_stat==w_stat else np.nan,
                         "wilcoxon_p": float(w_p) if w_p==w_p else np.nan,
                         "note": "mean_diff<0 => GAT lower RMSE (better)"})
sig_df = pd.DataFrame(sig_rows)
sig_df.to_csv(os.path.join(WORK, "step5_significance.csv"), index=False)
print(sig_df.to_string())
print("\nInterpretation rule: only call a difference real if mean_diff<0 AND p<0.05 on BOTH tests.")
for _, row in sig_df.iterrows():
    if row["n"] >= 3 and row["mean_diff_rmse"] == row["mean_diff_rmse"]:
        real = (row["mean_diff_rmse"] < 0) and (row["t_p"] < 0.05) and (not (row["wilcoxon_p"]==row["wilcoxon_p"]) or row["wilcoxon_p"] < 0.05)
        print(f"  {row['resource']:11s} {row['comparison']:20s}: "
              f"{'GAT significantly better' if real else 'no significant improvement (null/!=)'} "
              f"(mean_diff_rmse={row['mean_diff_rmse']:.4f}, t_p={row['t_p']:.3g})")


STEP 5 -- paired significance tests (per-site errors)
      resource             comparison     n  mean_diff_rmse     t_stat           t_p  wilcoxon_stat    wilcoxon_p                                    note
0       carbon      gat_vs_unweighted  4887       -0.003605  -0.801219  4.230438e-01      2164128.5  9.665762e-01  mean_diff<0 => GAT lower RMSE (better)
1       carbon  gat_vs_random_control  4887       -1.709548 -13.618555  1.761541e-41      5503962.0  2.348462e-06  mean_diff<0 => GAT lower RMSE (better)
2  electricity      gat_vs_unweighted  4887        2.370124   4.377309  1.226616e-05      2353257.0  1.800709e-02  mean_diff<0 => GAT lower RMSE (better)
3  electricity  gat_vs_random_control  4887     -700.974789 -42.905865  0.000000e+00      1702643.0  0.000000e+00  mean_diff<0 => GAT lower RMSE (better)
4        water      gat_vs_unweighted  4887       -0.020046  -5.811459  6.585889e-09       320362.0  5.565277e-18  mean_diff<0 => GAT lower RMSE (better)
5        water  gat_v

In [49]:
section("STEP 5 -- diagram view: best-single / simple-average / graph-weighted (+ random check)")
d = df_unweighted.copy()
d["region_id"] = d.apply(lambda row: region_id_of(row["resource"], int(row["site_id"])), axis=1)
m = d.merge(actuals, on=["region_id", "resource", "timestamp"], how="inner")
per_model = m.groupby(["site_id", "resource", "model"]).apply(_metrics).reset_index()
best_single = per_model.loc[per_model.groupby(["site_id", "resource"])["RMSE"].idxmin()]

versions = {
    "best_single_model(ORACLE)": best_single,
    "simple_average":  met_all[met_all.condition == "unweighted"],
    "graph_weighted":  met_all[met_all.condition == "gat_weighted"],
    "random_edge_check": met_all[met_all.condition == "random_control"],
}
rows = []
for name, dfm in versions.items():
    for r in sorted(met_all.resource.unique()):
        sub = dfm[dfm.resource == r]
        rows.append({"version": name, "resource": r,
                     "RMSE": round(sub["RMSE"].mean(), 4), "MAE": round(sub["MAE"].mean(), 4)})
diagram_tbl = pd.DataFrame(rows)
diagram_tbl.to_csv(os.path.join(WORK, "step5_diagram_versions.csv"), index=False)
print(diagram_tbl.pivot(index="resource", columns="version", values="RMSE").to_string())
print("\nRandom-edge check: compare 'graph_weighted' vs 'random_edge_check' above. If they are ~equal, the "
      "co-location signal is NOT adding anything over random edges — reported as information, not a failure.")


STEP 5 -- diagram view: best-single / simple-average / graph-weighted (+ random check)
version      best_single_model(ORACLE)  graph_weighted  random_edge_check  simple_average
resource                                                                                 
carbon                         25.1836         27.7454            29.4550         27.7490
electricity                   918.8964       1078.1668          1779.1416       1075.7967
water                          36.6690         42.1266            41.7146         42.1466

Random-edge check: compare 'graph_weighted' vs 'random_edge_check' above. If they are ~equal, the co-location signal is NOT adding anything over random edges — reported as information, not a failure.


## Final summary

In [50]:
section("FINAL SUMMARY (generated from this run's variables — not hardcoded)")
print(f"Models run, in order: {' -> '.join(MODELS)}")
print(f"Resources forecast: {list(RESOURCES)} (water added this version; water_stress/temperature remain non-forecastable)")
print(f"RL: not present in this notebook (removed per CCAI scope decision).\n")

print("Coverage:")
print(cov_df.to_string(index=False))

print("\nPer-resource aggregate RMSE (unweighted / gat_weighted / random_control):")
print(agg["RMSE"].unstack("condition").round(4).to_string())

print("\nSignificance verdicts (gat_weighted vs unweighted, per resource):")
for _, row in sig_df[sig_df.comparison == "gat_vs_unweighted"].iterrows():
    if row["n"] >= 3 and row["mean_diff_rmse"] == row["mean_diff_rmse"]:
        real = (row["mean_diff_rmse"] < 0) and (row["t_p"] < 0.05) and (not (row["wilcoxon_p"]==row["wilcoxon_p"]) or row["wilcoxon_p"] < 0.05)
        print(f"  {row['resource']:11s}: {'SIGNIFICANT improvement' if real else 'no significant improvement'} "
              f"(mean_diff_rmse={row['mean_diff_rmse']:.4f}, t_p={row['t_p']:.3g}, n={int(row['n'])})")
    else:
        print(f"  {row['resource']:11s}: not testable ({row['note']})")

print("\nAll files written to /kaggle/working/ — see Section 9 for the list.")


FINAL SUMMARY (generated from this run's variables — not hardcoded)
Models run, in order: SARIMA -> Nexus
Resources forecast: ['electricity', 'carbon', 'water'] (water added this version; water_stress/temperature remain non-forecastable)
RL: not present in this notebook (removed per CCAI scope decision).

Coverage:
 model    resource  region_series_total  forecast_ok  failed_or_skipped
SARIMA electricity                   94           94                  0
SARIMA      carbon                   94           94                  0
SARIMA       water                   24           24                  0
 Nexus electricity                   94           94                  0
 Nexus      carbon                   94           94                  0
 Nexus       water                   24           24                  0

Per-resource aggregate RMSE (unweighted / gat_weighted / random_control):
condition    gat_weighted  random_control  unweighted
resource                                         